# Confusion Matrix

## Imports

In [1]:
# IMPORTS
import os
import shutil
import time
import sys
import random

import numpy as np
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mpcol

from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FormatStrFormatter

from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pandas as pd
from dython.nominal import associations
from dython.nominal import identify_nominal_columns

from scipy import stats
from sklearn import datasets, mixture

from termcolor import colored, cprint
# Termcolor guide: https://pypi.org/project/termcolor/

from openpyxl import Workbook
from openpyxl import load_workbook

from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV

import seaborn as sns

# %matplotlib widget
%matplotlib inline
# %matplotlib notebook

# Try to use this website to use the explode feature, so we can see internal blocks and space everything out
# https://terbium.io/2017/12/matplotlib-3d/ 

# Use this website to make your GIFs - generally 50 delay per frame is good
# https://ezgif.com/maker



/Users/liamroy/miniforge3/envs/phd/lib/python3.9/site-packages/dython/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution


In [2]:
%pwd

'/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/scripts'

## Create the Matrix

In [3]:
# Matrix Data Setup

excel_filepath = '/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/data/proxy_validation/proxy_validation.xlsx'
excel_sheetname = 'PROXY_FINAL_PILOT'

actual_class = ['WFI', 'AO', 'FO', 'NH', 'C']
predicted_class = ['WFI', 'AO', 'FO', 'NH', 'C', 'NON']

save_path = '/Users/liamroy/Documents/Studies/Monash_31194990/PHD/Studies/Study_04/LLM_vocab_optimization/plots/conf_matrix/'

plotter = None # None or True

In [4]:
def getConfusionMatrix(excel_file_path, excel_file_sheet, row_idx_start, row_idx_end, colum_idx, actual_classes, predicted_classes):
    # Number of classes is N
    # Creates the N x M grid for the predictions for each class
    # Rows are actual classes, columns are predicted classes
    df = pd.read_excel(excel_file_path, sheet_name = excel_file_sheet)

    data = df.iloc[row_idx_start-2:row_idx_end-1, colum_idx-1].values  # This gives you a 1D array of 30 elements
    
    # Round the percentages down to nearest iteger and convert to integers
    data_numeric = np.nan_to_num(data.astype(float))
    confMat_percent = np.round(data_numeric * 100).astype(int)
    # Reshape the data into desired shape
    # Example: if you want 5 rows and 6 columns
    confMatrix = confMat_percent.reshape(len(actual_classes), len(predicted_classes))

    # Print the reshaped data
    # print(confMatrix)

    # Create a confusion matrix from the reshaped data
    return confMatrix

In [5]:
human_resp_LLM_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=2, 
                                       row_idx_end=31, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Human Response for LLM Cond - confMat:\n{human_resp_LLM_cond_confMat}\n\n")

proxy_resp_LLM_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=2, 
                                       row_idx_end=31, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Proxy Response for LLM Cond - confMat:\n{proxy_resp_LLM_cond_confMat}\n\n")

human_resp_human_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=32, 
                                       row_idx_end=61, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Human Response for Human Cond - confMat:\n{human_resp_human_cond_confMat}\n\n")

proxy_resp_human_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=32, 
                                       row_idx_end=61, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Proxy Response for Human Cond - confMat:\n{proxy_resp_human_cond_confMat}\n\n")

human_resp_random_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=62, 
                                       row_idx_end=91, 
                                       colum_idx=1, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Human Response for Rand Cond - confMat:\n{human_resp_random_cond_confMat}\n\n")

proxy_resp_random_cond_confMat = getConfusionMatrix(excel_file_path=excel_filepath, 
                                       excel_file_sheet=excel_sheetname, 
                                       row_idx_start=62, 
                                       row_idx_end=91, 
                                       colum_idx=2, 
                                       actual_classes=actual_class, 
                                       predicted_classes=predicted_class)
print(f"Proxy Response for Human Cond - confMat:\n{proxy_resp_random_cond_confMat}\n\n")


Human Response for LLM Cond - confMat:
[[73  2  2  5  7 12]
 [ 8 33 50  3  3  2]
 [ 2 57 40  0  0  2]
 [15 10  3 52 12  8]
 [ 5  8  0 52 32  3]]


Proxy Response for LLM Cond - confMat:
[[70  5  0  0  0 25]
 [20 40 30  0  0 10]
 [ 0 70 20  0  0 10]
 [15  0  0 40 40  5]
 [10  0  0 35 30 20]]


Human Response for Human Cond - confMat:
[[65  8  8  5  7  7]
 [ 0 55 43  0  2  0]
 [ 0 50 48  2  0  0]
 [57  3  5 18 15  2]
 [60  2  3  7 27  2]]


Proxy Response for Human Cond - confMat:
[[60 15  0 10  0 15]
 [ 0 70 20  0  0 10]
 [10 60 15  0  5 10]
 [ 5  0 10 65 15  5]
 [30  5  0 15 45  5]]


Human Response for Rand Cond - confMat:
[[ 2 15 25 15 38  5]
 [13  2  3 58 12 12]
 [ 0 30 17 28 23  2]
 [ 0 45 15  7 23 10]
 [15  0  5 57 18  5]]


Proxy Response for Human Cond - confMat:
[[ 0 50  5 30 10  5]
 [ 5  0  0 65 15 15]
 [ 0 35 10 20 15  5]
 [ 0 45 35 10 10  0]
 [ 0 15  0 55 25  0]]




In [6]:

def plot_matrix(matrix_list, condition_name_list, save_filename_list, save_path, type_of_matrix, real_or_proxy_list):

    """
    Plots confusion matrices from a list of matrices and saves them to specified paths.
    
    Args:
        matrix_list (list): List of confusion matrices to plot.
        condition_name_list (list): List of condition names corresponding to each matrix.
        save_filename_list (list): List of filenames to save the plots.
        save_path (str): Path where the plots will be saved.
        real_or_proxy_list (list): List indicating whether the state estimation is real or proxy.
        type_of_matrix (str): Type of matrix to plot, either 'conf_matrix' or 'delta_matrix'.
    """
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    for x in range(len(matrix_list)):

        save_str = save_path + save_filename_list[x]

        # Plotting Setup
        fig, axs = plt.subplots(figsize=(12, 12))
        
        if type_of_matrix == 'conf_matrix':
            # Title and Axes Labels
            plot_title = f'\n{real_or_proxy_list[x]} State Estimation Confusion\nMatrix for {condition_name_list[x]} Generated Poses\n'
            norm = mpcol.Normalize(vmin=0, vmax=100)
            im = axs.imshow(matrix_list[x], cmap='viridis', norm=norm)


        
        elif type_of_matrix == 'delta_matrix':
            # Title and Axes Labels
            plot_title = f'\nSigned Difference in State Estimation\n(Real Human – Proxy) for {condition_name_list[x]} Generated Poses\n'


            # colors = [
            #     (0.0, '#540000'),  # dark red
            #     (0.25, '#40190b'), # sienna (mild brown)
            #     (0.5, '#035400'),  # medium aquamarine (mild green)
            #     (0.75, '#40190b'), # sienna (mild brown)
            #     (1.0, '#540000')   # dark red
            # ]

            colors = [
                (0.0, '#fde724'), 
                (0.1, '#7ae64f'), 
                (0.2, '#1eaac4'), 
                (0.3, '#276cc6'),  
                (0.4, '#443e7f'), 
                (0.5, '#440154'),  
                (0.6, '#443e7f'),  
                (0.7, '#276cc6'),  
                (0.8, '#1eaac4'),  
                (0.9, '#7ae64f'),  
                (1.0, '#fde724'),  
            ]
           
            
            # Create custom colormap
            custom_colormap = LinearSegmentedColormap.from_list('custom_sep', [c[1] for c in colors])


            # Apply symmetric normalization
            norm = mpcol.TwoSlopeNorm(vmin=-100, vcenter=0, vmax=100)
            
            # Example use in imshow:
            im = axs.imshow(matrix_list[x], cmap=custom_colormap, norm=norm)
                    

        else:
            raise ValueError(f"Invalid type_of_matrix: {type_of_matrix}")




        # Title and Axes Labels
        axs.set_title(plot_title, size=32, weight='bold')

        # if type_of_matrix == 'conf_matrix':
        axs.set_xlabel(f"Robot State Selected by {real_or_proxy_list[x]}", size=32, weight='bold')
        # elif type_of_matrix == 'delta_matrix':
        #     axs.set_xlabel(f"Robot State Selected by Human/Proxy", size=32, weight='bold')
        
        axs.set_ylabel("True Robot State\n", size=36, weight='bold')

        # Show all ticks and label them with the respective list entries
        axs.set_yticks(np.arange(len(actual_class)), labels=actual_class)
        axs.set_xticks(np.arange(len(predicted_class)), labels=predicted_class)

        # Rotate the tick labels and set their alignment.
        plt.setp(axs.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor", size=28, weight='bold')
        plt.setp(axs.get_yticklabels(), rotation=45, ha="right", rotation_mode="anchor", size=28, weight='bold')

        # Make the Sidebar
        divider = make_axes_locatable(axs)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=26 )


        # Loop over data dimensions and create text annotations.
        for i in range(len(actual_class)):
            for j in range(len(predicted_class)):
                text = axs.text(j, i, f"{matrix_list[x][i, j]:.0f}%",
                                ha="center", va="center", color="w", size=32, weight='bold')

        fig.tight_layout()

        # Save the fig
        plt.savefig(save_str, bbox_inches='tight', pad_inches=0.25)

        # Show the fig if plotter is not set to None
        if plotter:
            plt.show()

        if plotter == None:
            print("Plots closed.")
            plt.close()



## Plotting Confusion Matrices

In [7]:
conf_matrix_list = [human_resp_LLM_cond_confMat, 
                    proxy_resp_LLM_cond_confMat, 
                    human_resp_human_cond_confMat, 
                    proxy_resp_human_cond_confMat, 
                    human_resp_random_cond_confMat, 
                    proxy_resp_random_cond_confMat]

conf_save_filename_list = [
    'human_resp_LLM_cond_confMat.png', 
    'proxy_resp_LLM_cond_confMat.png', 
    'human_resp_human_cond_confMat.png', 
    'proxy_resp_human_cond_confMat.png', 
    'human_resp_random_cond_confMat.png', 
    'proxy_resp_random_cond_confMat.png'
]

conf_condition_name_list = [
    'LLM', 
    'LLM', 
    'Human', 
    'Human', 
    'Random', 
    'Random'
]

conf_real_or_proxy_list = [
    'Human', 
    'Proxy', 
    'Human', 
    'Proxy', 
    'Human', 
    'Proxy'
]

plot_matrix(matrix_list=conf_matrix_list, 
            condition_name_list=conf_condition_name_list, 
            save_filename_list=conf_save_filename_list, 
            save_path=save_path, 
            type_of_matrix='conf_matrix', 
            real_or_proxy_list=conf_real_or_proxy_list)



Plots closed.
Plots closed.
Plots closed.
Plots closed.
Plots closed.
Plots closed.


## Now Create Plots to Show Difference

In [8]:
# Create a plot to show the difference between the proxy and real matrixies for the three conditions (GPT4o, human, random)

delta_matrix_GPT4o = human_resp_LLM_cond_confMat - proxy_resp_LLM_cond_confMat
delta_matrix_human = human_resp_human_cond_confMat - proxy_resp_human_cond_confMat
delta_matrix_random = human_resp_random_cond_confMat - proxy_resp_random_cond_confMat

print(delta_matrix_GPT4o, "\n")
print(delta_matrix_human, "\n")
print(delta_matrix_random, "\n")

delta_matrix_list = [delta_matrix_GPT4o, 
                     delta_matrix_human, 
                     delta_matrix_random]
   
delta_condition_name_list = [
    'LLM', 
    'Human', 
    'Random', 
]

delta_save_filename_list = [
    'LLM_delta.png', 
    'Human_delta.png', 
    'Random_delta.png', 
]

conf_real_or_proxy_list = [
    'User/Proxy',
    'User/Proxy', 
    'User/Proxy', 
]


plot_matrix(matrix_list=delta_matrix_list, 
            condition_name_list=delta_condition_name_list, 
            save_filename_list=delta_save_filename_list, 
            save_path=save_path, 
            type_of_matrix='delta_matrix',
            real_or_proxy_list=conf_real_or_proxy_list)


[[  3  -3   2   5   7 -13]
 [-12  -7  20   3   3  -8]
 [  2 -13  20   0   0  -8]
 [  0  10   3  12 -28   3]
 [ -5   8   0  17   2 -17]] 

[[  5  -7   8  -5   7  -8]
 [  0 -15  23   0   2 -10]
 [-10 -10  33   2  -5 -10]
 [ 52   3  -5 -47   0  -3]
 [ 30  -3   3  -8 -18  -3]] 

[[  2 -35  20 -15  28   0]
 [  8   2   3  -7  -3  -3]
 [  0  -5   7   8   8  -3]
 [  0   0 -20  -3  13  10]
 [ 15 -15   5   2  -7   5]] 

Plots closed.
Plots closed.
Plots closed.
